<a href="https://colab.research.google.com/github/hwasun-zip/Catalog-AutoPilot/blob/main/Catalog_AutoPilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 라이브러리 설치

# anthropic: Claude API를 호출하기 위한 공식 SDK
!pip install -q anthropic pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 16.1 MB/s eta 0:00:00


In [2]:
# import + API 키 불러오기
import anthropic
import pandas as pd
import json
import time
from google.colab import userdata

# API 키는 코드에 직접 입력하지 않고, Colab의 '보안 비밀(Secrets)' 기능으로 안전하게 불러옴
api_key = userdata.get('ANTHROPIC_API_KEY')

client = anthropic.Anthropic(api_key=api_key)

print('anthropic SDK version:', anthropic.__version__)
print('클라이언트 초기화 완료')

anthropic SDK version: 1.2.0
클라이언트 초기화 완료


In [3]:
# 연결 테스트
test_response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=100,
    messages=[
        {"role": "user", "content": "이 메시지가 보이면 'API 연결 성공'이라고만 답해줘."}
    ]
)

print(test_response.content[0].text)

API 연결 성공


In [4]:
# Project 1 결과물 불러오기

from google.colab import files

print('price_outlier_details.csv 파일을 업로드해주세요 (Catalog Triage 프로젝트 결과물)')
uploaded = files.upload()

price_outlier_details.csv 파일을 업로드해주세요 (Catalog Triage 프로젝트 결과물)


Saving price_outlier_details.csv to price_outlier_details.csv


In [5]:
# 데이터 확인

outlier_df = pd.read_csv('price_outlier_details.csv')

print('전체 이상치 건수:', len(outlier_df))
print()
print(outlier_df.head(3))
print()
print('--- confidence_level 분포 ---')
print(outlier_df['confidence_level'].value_counts())

전체 이상치 건수: 32199

   restaurant_id               restaurant_name             menu_name  \
0          54106                 Fire Ass Thai  Spicy Massaman Curry   
1          19112  Tsim Yung Chinese Restaurant  C 7. Moo Goo Gai Pan   
2          41586              Taqueria Chapala         Mojarra Frita   

  main_category  menu_price  category_avg_price  price_z_score  \
0  Asian Fusion      1395.0               11.98         122.52   
1       Chinese       925.0               11.97         117.18   
2       Mexican      1049.0                9.87          98.97   

   ratio_to_category_avg   suspected_cause        confidence_level  
0                  116.4  소수점 오류 의심 (÷100)  High Confidence (자동확정)  
1                   77.3  소수점 오류 의심 (÷100)  High Confidence (자동확정)  
2                  106.3  소수점 오류 의심 (÷100)  High Confidence (자동확정)  

--- confidence_level 분포 ---
confidence_level
Needs Review (수동검토)       21699
High Confidence (자동확정)    10500
Name: count, dtype: int64


In [6]:
# Needs Review 중 이상치 정도가 큰 상위 30건만 우선 요약 대상으로 선정
# (전수 처리는 비용/시간이 크므로, 실무처럼 '심각도 높은 것부터' 우선 처리하는 방식으로 설계)
needs_review = outlier_df[outlier_df['confidence_level'] == 'Needs Review (수동검토)'].copy()
priority_sample = needs_review.nlargest(30, 'price_z_score')

print('요약 대상 건수:', len(priority_sample))
print(priority_sample[['restaurant_name', 'menu_name', 'menu_price', 'category_avg_price', 'ratio_to_category_avg']].head(10))

요약 대상 건수: 30
                           restaurant_name  \
3                        palios pizza cafe   
4                         Nusr-Et (Dallas)   
5   Rabble-Rouser Chocolate &amp; Craft Co   
6                            Euro's Gyro's   
7                    Kau Ba Saigon Kitchen   
8                           Republic Roots   
9              Crave Bakery and Coffee Bar   
10                       Northern D' Lites   
11                Chicken Express - Irving   
12               Chicken Express - Bedford   

                                            menu_name  menu_price  \
3                                         Greek Bread      599.00   
4                       Gold Istanbul Steak (10.5 oz)      590.00   
5                                      The Nutty Club      275.00   
6                                             Pudding      450.00   
7   Domaine Denis Bachelet Gevrey-Chambertín 2016,...      345.60   
8                                7 - Days Cleanse Set      350.00 

In [7]:
# 프롬프트 설계 원칙
# 1. 역할 부여: "카탈로그 품질 담당자를 돕는 분석 보조"로 역할을 명확히
# 2. 구조화된 입력: 메뉴명, 가격, 카테고리 평균가, 배수를 명확히 제시
# 3. 원인 후보를 유도: "이게 오류인지 정상인지"를 무작정 판단하게 하지 말고, 가능성 있는 원인들을 제시하도록 (금박스테이크처럼 브랜드명에 힌트가 있는 경우도 있음)
# 4. 구조화된 출력: 나중에 데이터로 다시 정리해야 하니, JSON 형식으로 답변받기

def build_prompt(row):
    return f"""당신은 배달 플랫폼의 카탈로그 품질 담당자를 돕는 분석 보조입니다.
아래 메뉴는 SQL 이상탐지에서 "가격이 카테고리 평균 대비 비정상적으로 높다"고 플래깅되었지만,
자동으로는 원인을 판단하지 못해 검토가 필요한 상태입니다.

[메뉴 정보]
- 상점명: {row['restaurant_name']}
- 메뉴명: {row['menu_name']}
- 메뉴 카테고리: {row['main_category']}
- 등록 가격: ${row['menu_price']:.2f}
- 카테고리 평균가: ${row['category_avg_price']:.2f}
- 평균 대비 배수: {row['ratio_to_category_avg']:.1f}배

[요청]
이 메뉴가 "① 가격 오류일 가능성" 또는 "② 정상적으로 비싼 상품(고급 재료, 대용량, 세트메뉴, 주류 등)일 가능성" 중
어느 쪽에 가까운지 메뉴명과 상점명의 단서를 바탕으로 판단하고, 아래 JSON 형식으로만 답하세요.

{{"판정": "오류 의심" 또는 "정상 고가 추정" 또는 "판단 불가",
"근거": "한 문장으로 판단 근거",
"확신도": "높음" 또는 "중간" 또는 "낮음"}}
"""

# 프롬프트 미리보기 (첫 번째 케이스로 테스트)
sample_prompt = build_prompt(priority_sample.iloc[0])
print(sample_prompt)

당신은 배달 플랫폼의 카탈로그 품질 담당자를 돕는 분석 보조입니다.
아래 메뉴는 SQL 이상탐지에서 "가격이 카테고리 평균 대비 비정상적으로 높다"고 플래깅되었지만,
자동으로는 원인을 판단하지 못해 검토가 필요한 상태입니다.

[메뉴 정보]
- 상점명: palios pizza cafe
- 메뉴명: Greek Bread
- 메뉴 카테고리: Italian
- 등록 가격: $599.00
- 카테고리 평균가: $13.38
- 평균 대비 배수: 44.8배

[요청]
이 메뉴가 "① 가격 오류일 가능성" 또는 "② 정상적으로 비싼 상품(고급 재료, 대용량, 세트메뉴, 주류 등)일 가능성" 중
어느 쪽에 가까운지 메뉴명과 상점명의 단서를 바탕으로 판단하고, 아래 JSON 형식으로만 답하세요.

{"판정": "오류 의심" 또는 "정상 고가 추정" 또는 "판단 불가",
"근거": "한 문장으로 판단 근거",
"확신도": "높음" 또는 "중간" 또는 "낮음"}



In [8]:
# 실제 API 호출 테스트
response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=200,
    messages=[
        {"role": "user", "content": sample_prompt}
    ]
)

print(response.content[0].text)

```json
{
  "판정": "오류 의심",
  "근거": "Greek Bread는 일반적인 빵/브레드 메뉴로 $599는 단일 식품으로는 물리적으로 불가능한 가격이며, 상점명(피자 카페)과 메뉴명의 특성상 고급 재료나 세트 구성으로 정당화하기 어렵다.",
  "확신도": "높음"
}
```


In [9]:
# 배치 처리 함수 만들기
import re

def get_llm_judgment(row, client):
    """한 건의 메뉴에 대해 Claude API를 호출하고 JSON 응답을 파싱해서 반환"""
    prompt = build_prompt(row)
    try:
        response = client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=200,
            messages=[{"role": "user", "content": prompt}]
        )
        raw_text = response.content[0].text

        # 응답에서 JSON 부분만 추출 (```json ... ``` 감싸져 있을 수 있으므로)
        json_match = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group())
            return parsed.get('판정'), parsed.get('근거'), parsed.get('확신도')
        else:
            return '파싱실패', raw_text[:100], None
    except Exception as e:
        return '오류', str(e), None

# 먼저 3건만 테스트 (비용 절약을 위해 소규모로 먼저 검증)
test_batch = priority_sample.head(3).copy()
results = []

for idx, row in test_batch.iterrows():
    verdict, reason, confidence = get_llm_judgment(row, client)
    results.append({'menu_name': row['menu_name'], 'verdict': verdict, 'reason': reason, 'confidence': confidence})
    print(f"✓ {row['menu_name']}: {verdict} ({confidence})")
    time.sleep(1)  # API 호출 간 짧은 대기 (rate limit 방지)

print()
print(pd.DataFrame(results))

✓ Greek Bread: 오류 의심 (높음)
✓ Gold Istanbul Steak (10.5 oz): 정상 고가 추정 (높음)
✓ The Nutty Club: 오류 의심 (높음)

                       menu_name   verdict  \
0                    Greek Bread     오류 의심   
1  Gold Istanbul Steak (10.5 oz)  정상 고가 추정   
2                 The Nutty Club     오류 의심   

                                              reason confidence  
0  Greek Bread는 일반적인 빵/사이드 메뉴로 $599.00은 배달 플랫폼에서 ...         높음  
1  Nusr-Et은 세계적으로 유명한 럭셔리 스테이크 프랜차이즈이며, 'Gold Ist...         높음  
2  Coffee and Tea 카테고리에서 $275는 극단적으로 높으며, '클럽(Clu...         높음  


In [10]:
# 나머지 27건까지 전체 30건 배치 처리
all_results = []

for idx, row in priority_sample.iterrows():
    verdict, reason, confidence = get_llm_judgment(row, client)
    all_results.append({
        'restaurant_name': row['restaurant_name'],
        'menu_name': row['menu_name'],
        'menu_price': row['menu_price'],
        'category_avg_price': row['category_avg_price'],
        'ratio_to_category_avg': row['ratio_to_category_avg'],
        'llm_verdict': verdict,
        'llm_reason': reason,
        'llm_confidence': confidence
    })
    print(f"✓ ({len(all_results)}/30) {row['menu_name'][:30]}: {verdict}")
    time.sleep(1)

llm_results_df = pd.DataFrame(all_results)
print()
print('--- 전체 판정 분포 ---')
print(llm_results_df['llm_verdict'].value_counts())

✓ (1/30) Greek Bread: 오류 의심
✓ (2/30) Gold Istanbul Steak (10.5 oz): 정상 고가 추정
✓ (3/30) The Nutty Club: 오류 의심
✓ (4/30) Pudding: 오류 의심
✓ (5/30) Domaine Denis Bachelet Gevrey-: 정상 고가 추정
✓ (6/30) 7 - Days Cleanse Set: 정상 고가 추정
✓ (7/30) Assorted Sandwich Platter: 정상 고가 추정
✓ (8/30) Pizza Puff: 오류 의심
✓ (9/30) 100 Tenders Game Day Meal: 정상 고가 추정
✓ (10/30) Gameday 100 Tender Meal: 오류 의심
✓ (11/30) 100 Tenders Game Day Meal: 정상 고가 추정
✓ (12/30) Gameday 100 Tender Meal: 오류 의심
✓ (13/30) 100 Tenders Game Day Meal: 정상 고가 추정
✓ (14/30) 5-Day Cleanse: 정상 고가 추정
✓ (15/30) Group Meal 20: 정상 고가 추정
✓ (16/30) Group Meal 20: 정상 고가 추정
✓ (17/30) Bruno Clavelier Vosne-Romanee : 정상 고가 추정
✓ (18/30) 6 - Days Cleanse Set: 정상 고가 추정
✓ (19/30) Skateboard: 오류 의심
✓ (20/30) Skateboard: 오류 의심
✓ (21/30) Rib Eye, 12oz - 44 ct Case: 정상 고가 추정
✓ (22/30) Rib Eye, 12oz - 44 ct Case: 정상 고가 추정
✓ (23/30) Rib Eye, 12oz - 44 ct Case: 정상 고가 추정
✓ (24/30) Rib Eye, 12oz - 44 ct Case: 정상 고가 추정
✓ (25/30) Rib Eye, 12oz - 44 ct Case: 정상 고가 추정
✓ 

In [11]:
# 일관성 체크 + 확신도 분포 확인

# 확신도별 분포 확인 - 신뢰도 낮은 판정이 얼마나 있는지 파악
print('--- 확신도별 분포 ---')
print(llm_results_df['llm_confidence'].value_counts())
print()

# 동일 메뉴명인데 판정이 갈린 케이스 찾기
inconsistent = llm_results_df.groupby('menu_name')['llm_verdict'].nunique()
inconsistent_menus = inconsistent[inconsistent > 1].index.tolist()
print('--- 판정이 일관되지 않은 메뉴 ---')
print(llm_results_df[llm_results_df['menu_name'].isin(inconsistent_menus)][['restaurant_name', 'menu_name', 'llm_verdict', 'llm_confidence']])

--- 확신도별 분포 ---
llm_confidence
높음    22
중간     8
Name: count, dtype: int64

--- 판정이 일관되지 않은 메뉴 ---
Empty DataFrame
Columns: [restaurant_name, menu_name, llm_verdict, llm_confidence]
Index: []


In [12]:
# 확신도가 '중간'인 케이스 확인 - LLM이 스스로 판단이 애매하다고 인정한 것들
medium_confidence = llm_results_df[llm_results_df['llm_confidence'] == '중간']
print(medium_confidence[['restaurant_name', 'menu_name', 'llm_verdict', 'llm_reason']])

                                      restaurant_name  \
6                         Crave Bakery and Coffee Bar   
13                                        Rejuv Juice   
17                                     Republic Roots   
21              Ruby Tuesday (10419 Hull Street Road)   
26                         Bodija African Food Market   
27  California Pizza Kitchen (5665 North Centerpar...   
28  California Pizza Kitchen (5665 North Centerpar...   
29   California Pizza Kitchen (7939 L. Tysons Corner)   

                              menu_name llm_verdict  \
6             Assorted Sandwich Platter    정상 고가 추정   
13                        5-Day Cleanse    정상 고가 추정   
17                 6 - Days Cleanse Set    정상 고가 추정   
21           Rib Eye, 12oz - 44 ct Case    정상 고가 추정   
26  Sunfolk Whole Peeled Straw Mushroom       오류 의심   
27   LARGE - CPK CLASSICS PASTA PACKAGE    정상 고가 추정   
28  LARGE - CPK FAVORITES PASTA PACKAGE    정상 고가 추정   
29   LARGE - CPK CLASSICS PASTA PACKAGE    정상 

In [13]:
# Step 1: 결과를 보기 좋게 정리 + CSV로 저장

# 리포트용 컬럼 정리 및 이름 다듬기
report_df = llm_results_df.copy()

# 컬럼 순서 및 이름을 리포트용으로 정리
report_df = report_df[[
    'restaurant_name', 'menu_name', 'menu_price', 'category_avg_price',
    'ratio_to_category_avg', 'llm_verdict', 'llm_confidence', 'llm_reason'
]].rename(columns={
    'restaurant_name': '상점명',
    'menu_name': '메뉴명',
    'menu_price': '등록가격($)',
    'category_avg_price': '카테고리평균가($)',
    'ratio_to_category_avg': '평균대비배수',
    'llm_verdict': 'AI판정',
    'llm_confidence': 'AI확신도',
    'llm_reason': 'AI판단근거'
})

# 판정 결과에 따라 '권장 액션' 컬럼 추가 (실무 워크플로우처럼)
def recommend_action(row):
    if row['AI판정'] == '오류 의심' and row['AI확신도'] == '높음':
        return '즉시 수정 요청 (담당자 확인)'
    elif row['AI판정'] == '오류 의심':
        return '검토 후 수정 (확신도 중간)'
    elif row['AI확신도'] == '중간':
        return '참고용 - 사람 최종 확인 필요'
    else:
        return '조치 불필요 (정상 판정)'

report_df['권장액션'] = report_df.apply(recommend_action, axis=1)

print('--- 리포트 미리보기 ---')
print(report_df.head(10))

--- 리포트 미리보기 ---
                                      상점명  \
0                       palios pizza cafe   
1                        Nusr-Et (Dallas)   
2  Rabble-Rouser Chocolate &amp; Craft Co   
3                           Euro's Gyro's   
4                   Kau Ba Saigon Kitchen   
5                          Republic Roots   
6             Crave Bakery and Coffee Bar   
7                       Northern D' Lites   
8                Chicken Express - Irving   
9               Chicken Express - Bedford   

                                                 메뉴명  등록가격($)  카테고리평균가($)  \
0                                        Greek Bread   599.00       13.38   
1                      Gold Istanbul Steak (10.5 oz)   590.00        8.75   
2                                     The Nutty Club   275.00        5.97   
3                                            Pudding   450.00       10.42   
4  Domaine Denis Bachelet Gevrey-Chambertín 2016,...   345.60       10.96   
5                        

In [14]:
# Step 2: 요약 통계 출력 (담당자가 맨 위에서 볼 요약)

print('=' * 50)
print('📋 Catalog AutoPilot 이상치 원인 분석 리포트')
print('=' * 50)
print(f"분석 대상: SQL 탐지 이상치 중 원인 불명(Needs Review) 상위 {len(report_df)}건")
print()
print(f"🔴 오류 의심: {(report_df['AI판정']=='오류 의심').sum()}건")
print(f"🟢 정상 고가 추정: {(report_df['AI판정']=='정상 고가 추정').sum()}건")
print()
print(f"즉시 수정 요청 대상: {(report_df['권장액션']=='즉시 수정 요청 (담당자 확인)').sum()}건")
print(f"사람 최종 확인 필요: {(report_df['권장액션']=='참고용 - 사람 최종 확인 필요').sum()}건")

📋 Catalog AutoPilot 이상치 원인 분석 리포트
분석 대상: SQL 탐지 이상치 중 원인 불명(Needs Review) 상위 30건

🔴 오류 의심: 9건
🟢 정상 고가 추정: 21건

즉시 수정 요청 대상: 8건
사람 최종 확인 필요: 7건


In [15]:
# Step 3: CSV로 저장 + 다운로드

from google.colab import files

report_df.to_csv('catalog_autopilot_report.csv', index=False, encoding='utf-8-sig')
print('저장 완료: catalog_autopilot_report.csv')

files.download('catalog_autopilot_report.csv')

저장 완료: catalog_autopilot_report.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>